# 🚀 Tool Kéo Anime Sạch Vào Google Drive 5TB (Zero Ads / 1080p)
**Tác dụng:**
- Kéo phim từ magnet link / torrent sạch (Nyaa.si, SubsPlease, Erai-raws, Blu-Ray).
- Mạng Google Colab tải siêu tốc (50MB/s - 100MB/s).
- Tự lưu thẳng vào thư mục `KhangFlix_Anime` trên Google Drive 5TB.
- Tự động xuất mã JSON danh sách tập để paste thẳng vào web KhangFlix!

### Bước 1: Kết nối Google Drive 5TB và cài đặt thư viện tải

In [ ]:
# Kết nối Google Drive
from google.colab import drive
import os, time

drive.mount('/content/drive')

# Thư mục lưu anime sạch trên Google Drive
DRIVE_DIR = '/content/drive/MyDrive/KhangFlix_Anime'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"[OK] Thư mục lưu: {DRIVE_DIR}")

# Cài đặt libtorrent và công cụ xử lý video siêu nhẹ
!apt install -y python3-libtorrent ffmpeg
print("[OK] Môi trường sẵn sàng!")

### Bước 2: Paste link Magnet hoặc tải file Torrent anime sạch

In [ ]:
import libtorrent as lt
import time, sys

# PASTE LINK MAGNET ANIME VÀO ĐÂY (Lấy từ Nyaa.si hoặc SubsPlease)
MAGNET_LINK = "magnet:?xt=urn:btih:EXAMPLE..." 

ses = lt.session()
ses.listen_on(6881, 6891)
params = {
    'save_path': DRIVE_DIR,
    'storage_mode': lt.storage_mode_t(2),
}

print("[*] Đang thêm torrent...")
handle = lt.add_magnet_uri(ses, MAGNET_LINK, params)
ses.start_dht()

print("[*] Đang tải metadata...")
while not handle.has_metadata():
    time.sleep(1)
print("[+] Đã tìm thấy file! Bắt đầu tải siêu tốc vào Google Drive...")

while handle.status().state != lt.torrent_status.seeding:
    s = handle.status()
    state_str = ['queued', 'checking', 'downloading metadata', \
                 'downloading', 'finished', 'seeding', 'allocating']
    print(f"\rTiến độ: {s.progress * 100:.1f}% | Tốc độ: {s.download_rate / 1000000:.1f} MB/s | Peers: {s.num_peers}", end="")
    sys.stdout.flush()
    time.sleep(2)

print("\n\n🎉 TẢI HOÀN TẤT VÀO GOOGLE DRIVE 5TB!")

### Bước 3: Tự động trích xuất Google Drive ID tạo danh sách tập cho KhangFlix

In [ ]:
from googleapiclient.discovery import build
from google.colab import auth
import json

auth.authenticate_user()
drive_service = build('drive', 'v3')

# Quét các file video trong thư mục KhangFlix_Anime
results = drive_service.files().list(
    q="mimeType contains 'video/' and trashed = false",
    fields="files(id, name, size)"
).execute()

items = results.get('files', [])
print(f"Tìm thấy {len(items)} file video anime trong Drive:\n")

episodes_json = []
for ep in items:
    print(f"- {ep['name']} -> ID: {ep['id']}")
    episodes_json.append({
        "name": ep['name'].replace('.mp4', '').replace('.mkv', ''),
        "driveId": ep['id'],
        "url": "",
        "archiveId": "",
        "file": ""
    })

print("\n--- COPY ĐOẠN JSON NÀY DÁN VÀO KHANGFLIX ---")
print(json.dumps(episodes_json, ensure_ascii=False, indent=2))